# Welcome to lkdata!

In this notebook, we will cover the basic properties of lkdata, as well as examples of how to use the built-in functionality. 

The `lkdata` objects allow us to manipulate the data in a number of ways, including binning by time, slicing, folding, data aggregation and mathematical transformations. 

We start by introducing the data classes using simulated data in sections 1 and 2. If you are interested in how the lkdata structure can be applied to real data, you can check out the [example notebook](./nb2_lkdata_example.ipynb). 


# 1 - Handling 1-Dimensional timeseries data

In the simplest case, with only time and data, there are few benefits over simply using pandas, but astronomical data is more complicated. At the very least we should like to be able to handle uncertainties with numerical data.

Before we begin, let's create a random dataset to learn about the structure.

In [ ]:
# Import all of the packages we will use throughout this notebook
import numpy as np
import matplotlib.pyplot as plt
import lkdata as ld

print(ld.__path__)

In [ ]:
# Generate some random times.
times = np.append(np.linspace(0, 24, 120), np.linspace(26, 40, 120))

# Generate some random data
data = np.random.standard_normal(len(times))

# Generate some errors
data_err = np.abs(data) * 0.1

### 1.1 DataSeries

The `DataSeries` object can be used to represent any series of numerical data associated with time. In the `lkdata` ecosystem, data are unitless python data types, but metadata for units can be stored and accessed quite easily.

In [ ]:
# Create a DataSeries object with data, associated uncertainties, and time information, and extra 'metadata'
series = ld.DataSeries(
    data=data,
    uncertainty=data_err,
    time_indices={"time": times, "offset_time": times + 200},
    extra_metadata="This is my first DataSeries",
    time_units="days",
)
series

Great, we've initialized our first DataSeries! The representation of the object is very similar to a pandas DataFrame. The above cell shows that this is a DataSeries object with a length of 120 with non-zero uncertainties. A default ranged time_index has been created and all provided time columns are included as a [MultiIndex](https://pandas.pydata.org/docs/user_guide/advanced.html). Metadata are not shown in the default [repr](https://docs.python.org/3/library/functions.html#repr), but you can see more information using describe_series(), as shown below.

In [ ]:
series.describe_series()

You can access any of the properties, including the different time arrays and metadata, by using their keynames.

In [ ]:
series.offset_time[:10]

In [ ]:
series.time_units

### 1.2 BoolSeries and BitwiseSeries

Creating a BoolSeries or a BitwiseSeries is very similar to creating a DataSeries. The main difference is the data type.

We will create one my making a boolean array based on the random data generated in [Section 1.1](#11-dataseries), selecting out only positive data as an example.

In [ ]:
# Create a boolean array based on the data values
mask = data > 0
mask

In [ ]:
boolseries = ld.BoolSeries(
    data=mask,
    uncertainty=data_err,
    time_indices={"time": times},
)
boolseries.describe_series()

And just like that we have a `BoolSeries`!

In this case, we assigned an uncertainty value as we did for the `DataSeries`. However, the numerical uncertainties do not apply to boolean values so the uncertainty values are considered metadata, not true Uncertainty values. Modifying or slicing the BoolSeries with not apply to the given uncertainty as they are treated strictly as additional metadata.

`BitwiseSeries` was developed to handle the [TESS](https://outerspace.stsci.edu/display/TESS/2.0+-+Data+Product+Overview) and [Kepler/K2](https://archive.stsci.edu/files/live/sites/mast/files/home/missions-and-data/k2/_documents/MAST_Kepler_Archive_Manual_2020.pdf#page=18) quality masks. The quality flags are stored in a bitwise operation such that each flag is represented by a power of 2. A quality value of 3 would be composed of the flags 1 ($2^0$) and 2 ($2^1$). A quality value of 4 would only be the flag 4 ($2^2$).

In [ ]:
flags = np.random.choice(16, size=len(times))
code_dict = {2**i: f"code {2**i}" for i in range(5)}
bitseries = ld.BitwiseSeries(
    flags,
    time_indices={"time": times},
    code_dict=code_dict,
    display_as="int",  # try "bitset" and "detailed"
)
bitseries

Note the additional keyword arguments `code_dict` and `display_as`. 

The `display_as` keyword can be "int", "bitset", or "detailed" (if a code dictionary is provided, otherwise it is the same as "bitset"). The "int" display shows the values as a combination of the powers of 2, as they would be given by TESS or Kepler. The "bitset" display shows the set of all included powers of 2.


Providing `code_dict` enables the detailed display, which could provide a description of the flags or other metadata to be associated with particular flags.

The values in a `BitSeries` are converted to a custom `BitSet` object, a `set` subclass which splits integer values into powers of 2 and enables bitwise logic and math.

In [ ]:
from lkdata.bitset import BitSet

In [ ]:
BitSet(10)

In [ ]:
BitSet(10) + 3

In [ ]:
2 in BitSet(10)

## 1.3 Manipulating DataSeries data

In [ ]:
print(f"The series has shape {series.shape}")
downsampled_series = series.downsample(level="time")  # default 5 time steps
print(f"The downsampled series has shape {downsampled_series.shape}")
print(" ")
downsampled_series

### 1.3.1 Downsampling and binning

In this case, the size of the time steps changed before and after the 'gap' in time. Downsample can handle gaps in time, but does expect uniform time steps when time data is available. In this case, only data before the gap is binned (you can see this by looking at the index values and length of downsampled_series from the cell above). 

You can force binning of the data by specifying level='time_index'. This forces downsample to use the index values rather than the time values themselves. The downside of this is that the bins before and after the gap have different effective exposure times.

In [ ]:
downsampled_series = series.downsample(level="time_index")  # default 5 time_index steps
print(f"The series has shape {series.shape}")
print(f"The downsampled series has shape {downsampled_series.shape}")
downsampled_series

Downsample ensures that only bins with the given number of frames are returned. Downsample uses a default aggregation method for each data type, summing numerical data, adding uncertainties in quadrature, using logical or for boolean, and bitset addition for bitwise data. However an arbitrary bin function is also available for which the aggregation function can be set (as well as the uncertainty aggregation function for numerical data). The first argument of the bin function defines the left edges of each bin.

In [ ]:
series.bin(
    np.linspace(0, 24, 10),
    agg_func="mean",
    uncertainty_agg_func=lambda arr: np.sqrt(np.mean(arr**2)),
    level="time",
)

### 1.3.2 - slicing

DataSeries objects can be easily sliced in the same manner you [slice numpy arrays](https://numpy.org/doc/stable/user/absolute_beginners.html#indexing-and-slicing). In the example below we slice every other sample from the first 100 time steps in the DataSeries.

In [ ]:
sliced_series = series[:100:2]
print(f"The series has shape {series.shape}")
print(f"The series after slicing has shape {sliced_series.shape}")
sliced_series

In [ ]:
# Let's take a look at what these DataSeries look like
fig, ax = plt.subplots(1)
ax.set_title("Example of DataSeries binning and slicing")
ax.plot(series.time, series.data, label="original data")
# Note we are artificially adding a y-offset to the data for clarity
ax.plot(
    downsampled_series.time, downsampled_series.data - 5, label="downsampled data"
)  # bin 5 datapoints
ax.plot(
    sliced_series.time, sliced_series.data - 10, label="sliced data"
)  # First 100 time steps, keeping every other step
ax.set_xlabel(f"Time ({series.time_units})")
ax.legend()
plt.show()

# 2 - Handling timeseries for 2-Dimensional data, data cubes

Now let's take a look at timseries of 2-D images. lkdata handles this type of data with the DataCube class. For TESS, Kepler, and K2 data, this would be the data structure used to describe image timeseries, such as TPFs and FFI data. The DataCube structure allows us to manipulate this data in a number of ways, including binning (either spatially or by time), slicing, folding, or doing mathematical operations. We will look at these in more detail in the following cells. 

Before we begin, let's create a random dataset to learn about the structure. We will apply this to real data in section 3 below. 


## 2.1 creating a DataCube

In [ ]:
# Generate some random times.
times = np.linspace(0, 24, 120)

# Generate some random data
data = np.random.randn(len(times), 5, 7)

# Generate some errors
data_err = np.abs(data) * 0.1

In [ ]:
# print out the first 'image'
fig, ax = plt.subplots(2, tight_layout=True)
fig.colorbar(ax[0].imshow(data[0]))
fig.colorbar(ax[1].imshow(data_err[0]))
ax[0].set_title("data")
ax[1].set_title("data error");

In [ ]:
print(times.shape, data.shape, data_err.shape)

We have now generated the data needed to make a DataCube. In this example, we are making a timeseries with 120 time steps. We also have a 2-dimentional data product for each of these times steps. We can now put this information into our DataCube object. 

In [ ]:
cube = ld.DataCube(
    data=data,
    uncertainty=data_err,
    time_indices={"time": times},
    meta="My first DataCube",
)

In [ ]:
cube

Great! You can see that when we call the cube object, we get a basic summary of the contents. This tells us the shape of the cube and whether there are uncertainties associated with the data. It also shows a visualization of the data contained in the first time step.

We can get more information about the data contained in the Cube by calling describe_cube()

In [ ]:
cube.describe_cube()

## 2.2 Manipulating DataCubes - a bit like numpy, a bit like pandas

As with the DataSeries, you can manipulate the data cubes by slicing or binning (either temporally or spatially) the data. We demonstrate these functions in the cells below. 

### 2.2.1 Aggregating like pandas

`lkdata` objects inherit a lot of functionality from pandas objects. For example, you can call mean/median/min/max functions specifying the axis as you would a pandas dataframe. 

Aggregation methods, like `mean` and `median` operate on the time or space axes. Axis 0 being time, and axis 1 being space (both row and column).

Notice in the case below, you get a tuple as a result. When aggregating over time, we are no longer dealing with time series and the returned product is either a tuple containing arrays for data and errors, or just the array for data.

In [ ]:
data_mean, error_mean = cube.mean(axis=0)
data_mean.shape

In [ ]:
plt.colorbar(plt.imshow(data_mean));

Aggregating over space compresses the data and a `Series` is returned.

In [ ]:
cube.mean(axis=1)

#### 2.2.2 - Slicing like numpy
Now that we have our DataCube set up, we can start manipulating the data as we did for DataSeries objects. 

For example, what if we want only 1 out of every 10 time steps?

In [ ]:
sub_cube = cube[::10, :, :]  # Slice in the time step dimension
sub_cube

Or only the last time step?

In [ ]:
cube[-1, :, :]

Or to keep all timesteps for a specific subset of pixels?

In [ ]:
sub_cube = cube[:, 1:4, 2:5]  # Cut out a selection of pixels
sub_cube

Non-continuous spatial cuts result in `SeriesCollection` objects, see [Section 3](#3---seriescollections).

### 2.3 Downsampling (binning) by time

Instead of slicing, we may instead want to bin the data in the time dimension. This has the effect of taking the *sum* of each 'pixel' over the specified number of time steps. Note that the uncertaintaies are automatically added in quadrature. 

As with the DataSeries in Section 1.3.1, you can specify the specific value in the time index you want to include. 

In [ ]:
# Downsample every 5 time steps. If nframes is unspecified, 5 is the default value.
timebin_cube = cube.downsample(nframes=5, level="time")
timebin_cube

In [ ]:
# Verify that the uncertainties were added in quadrature
print(cube.uncertainty[0:5, 0, 0])  # first 5 uncertainty values for pixel (0,0)
print(
    timebin_cube.uncertainty[0, 0, 0]
)  # first binned uncertainty value for pixel (0,0)

np.sqrt(np.sum(cube.uncertainty[0:5, 0, 0].array ** 2))

### 2.4 Downsampling (binning) spatially

You are also allowed to spatially downsample DataCube objects. This would create the *sum* of pixels. If you specify "factor={integer}", it will use the same downsample size for column and row. For unevenly sized bins, you are able to sepcify col_factor/row_factor. Column/rows at the edge that do not "fill up" a new bin are discarded. In the case below, this would mean that column 6 and row 4 are discarded in the resulting DataCube object. 

TODO: enable downsample command to accept mean or median (default median is fine). 

In [ ]:
spacebin_cube = cube.spatial_downsample(factor=2)
spacebin_cube

In [ ]:
spacebin_cube = cube.spatial_downsample(col_factor=3, row_factor=2)
spacebin_cube

# 3 - SeriesCollections

But what happens when you extract non-contiguous pixels from a `Cube`?

In [ ]:
sub_cube = cube[:, ::2, ::2]  # Cut out all time steps for every other pixel
sub_cube.describe_collection()

This looks a little bit different. Since we no longer have contiguous blocks of pixels, this now returns the underlying pandas data frame contianing information on each pixel. Note that in this case you can no longer treat the object as a DataCube with the same functionality as before. Instead the data is handled by the third class, a `SeriesCollection`, which as the name implies is a collection of individual light curves that we want to treat as a cohesive unit, but are not spatially tied together. 

You are also able to generate your own DataSeriesCollections, as shown in the cell below. 

In [ ]:
# Generate some random data for 10 'timeseries' with 120 time steps
data = np.random.randn(120, 10)

# Generate some errors
data_err = np.abs(data) * 0.1

collection = ld.DataSeriesCollection(
    data=data,
    uncertainty=data_err,
)

collection

In [ ]:
collection.describe_collection()

# Conclusion

lkdata provides a way to quickly manipulate timeseries-like data. As lkdata is built on pandas DataFrames, transformations such as slicing and binning are easily (and quickly) accomplished. In addition, pandas provides a stable and well-developed backbone to build upon. 

This tutorial covered the basic features of lkdata, including the types of data products available as well as the types of manipulations you can do with that data. It also gave an introduction to how this can be done using real timeseries data from TESS. These examples serve as building blocks to construct larger and more powerful DataSets. 

The next tutorial puts these concepts into practice using data from the TESS mission. It also extends these concepts by introducing the DataSet class, which bundles any number of Series, Collections, or Cubes into one product for swift handling.